# Imports & Configuration

**What this cell does:** Silences Python `warnings` (e.g. deprecation or future warnings) so the notebook output only shows results.

**Core concept:** Warnings are emitted by the Python client, not by Spark jobs. Suppressing them does not affect Spark execution; it only keeps the output readable. In production code it is better to filter specific warning categories instead of all of them.

In [0]:
import warnings
warnings.filterwarnings("ignore")

**What this cell does:** Configures IPython to display the result of every expression in a cell, not just the last one.

**Core concept:** By default a notebook cell only renders the value of its final expression. Setting `ast_node_interactivity = 'all'` is handy for exploration (e.g. printing a partition count and a schema in the same cell).

In [0]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

**What this cell does:** Imports Spark SQL data types and the functions module (aliased as `F`) used for column expressions such as `F.col`, `F.split`, `F.sum`, `F.broadcast`.

**Core concept:** On Databricks, a `SparkSession` named `spark` is created automatically when the notebook attaches to a cluster, so there is no need to build one. This cluster uses standard access mode, which runs on **Spark Connect**: the Python code is a thin client that sends logical plans to the Spark driver over gRPC. Because of that, low-level APIs such as `spark.sparkContext` and `df.rdd` are not available; everything is done through the DataFrame / SQL API.

In [0]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
# SparkSession is pre-created on Databricks as `spark`; no need to import/build it

**What this cell does:** Sets the maximum input split size to 256 MB and defines `DATA_PATH`, the Unity Catalog volume that holds the input parquet files.

**Core concept:** Spark has two kinds of configuration:
* **Static (cluster-level)** settings such as `spark.driver.memory`, `spark.serializer` or `master` are fixed when the cluster starts. On Databricks they are set in the cluster's Spark config, not in a notebook.
* **Runtime (session-level) SQL** settings such as `spark.sql.files.maxPartitionBytes` can be changed at any time with `spark.conf.set`.

`maxPartitionBytes` controls how many bytes of file data go into one read partition. Bigger values mean fewer, larger read tasks; smaller values mean more, smaller tasks.

In [0]:
# On Databricks the `spark` session already exists and is attached to the cluster.
# Static settings (driver memory, serializer, master) are cluster-level: set them in the
# cluster's Spark config if needed. Only runtime SQL configs can be set here.
spark.conf.set("spark.sql.files.maxPartitionBytes", "268435456")  # 256 MB

# Data source: Unity Catalog volume
DATA_PATH = "/Volumes/lingaro_sandbox_ne/sandani_sandbox/volume_managed"

<h1> Topics </h1>

1. Reading Files (parquet)
2. Narrow Operations
   - `filter`
   - `withColumn`: adding/modifying a column
   - `select`: selecting relevant column
3. Wide Operations
   - Joins
     - Sort Merge Join
     - Broadcast Join
   - GroupBy
     - `count`
     - `sum`
     - `countDistinct`

**How to see the query DAG for each action in this notebook**

Each action (`show`, `write`, `count`) runs one or more Spark jobs. To inspect them:
* Expand **Spark Jobs** under the cell output, then open a job or click **View** next to the SQL query.
* Or open the cluster's **Spark UI** and go to the **SQL / DataFrame** tab to see the graph of physical operators (Scan, Filter, Project, Exchange, Sort, Join, HashAggregate).
* `df.explain("formatted")` prints the same physical plan as text without running the query.

Key things to look for: **Exchange** nodes mark a shuffle and a stage boundary, **WholeStageCodegen** boxes show operators fused into a single stage, and the row and size metrics on each node show how much data flows through it.

# Reading File

**What this cell does:** Defines a DataFrame on top of the `transactions.parquet` folder in the volume. This is the large fact table (~40M rows) used throughout the notebook.

**Core concept:** Spark is **lazy**. `spark.read.parquet` only reads metadata (schema from the parquet footers); no data is scanned until an *action* runs. Transformations build up a logical plan, and actions trigger execution of that plan as a DAG of stages and tasks. Parquet is a columnar format, so Spark can read only the columns a query needs (column pruning) and skip row groups using min/max statistics (predicate pushdown).

In [0]:
transactions_file = f"{DATA_PATH}/transactions.parquet"
df_transactions = spark.read.parquet(transactions_file)

**What this cell does:** Counts how many partitions the transactions DataFrame is split into when it is read.

**Core concept:** A **partition** is the unit of parallelism: each partition is processed by one task on one core. For file sources the partition count depends on total file size, `maxPartitionBytes`, `spark.sql.files.openCostInBytes` and the number of available cores. Too few partitions leave cores idle; too many add scheduling overhead.

The classic `df.rdd.getNumPartitions()` is not available on Spark Connect, so this cell tags each row with `spark_partition_id()` and counts the distinct ids. This counts only non-empty partitions and runs a small job, which you can see in the Spark UI.

In [0]:
# df.rdd is not available on standard access mode (Spark Connect); count non-empty partitions instead
df_transactions.select(F.spark_partition_id().alias("pid")).distinct().count()

# ------------------------------------------------------------------
# Why does Spark choose 9 read partitions?
# ------------------------------------------------------------------
# Actual data on disk (transactions.parquet):
#   • 163 parquet files, ~5–7.5 MB each
#   • 904,657,432 bytes ≈ 862.7 MB in total
#
# Settings involved:
#   • spark.sql.files.maxPartitionBytes = 268435456 (256 MB, set above)
#   • spark.sql.files.openCostInBytes   =   4194304 (4 MB default)
#   • defaultParallelism                = 8 (Standard_F8 single node, 8 cores)
#
# Step 1 - Add an open cost per file:
#   weighted size = data + numFiles * openCost
#                 = 862.7 MB + 163 * 4 MB = 1,514.7 MB (1,588,328,984 bytes)
#
# Step 2 - Compute the target split size:
#   bytesPerCore  = weighted size / defaultParallelism = 1,514.7 / 8 = 189.3 MB
#   maxSplitBytes = min(maxPartitionBytes, max(openCost, bytesPerCore))
#                 = min(256 MB, 189.3 MB) = 189.3 MB
#   -> the core count sets the limit here, not maxPartitionBytes.
#
# Step 3 - Pack the files into partitions:
#   No file exceeds 189.3 MB, so none is split. Files are sorted largest
#   first, and each partition is filled until the next file would push
#   it past 189.3 MB (each file counts as its size + 4 MB).
#   Result: 8 partitions of ~185-192 MB (17-25 files each) plus one
#   leftover partition of ~11.7 MB (2 files) = 9 partitions.
#
# Takeaways:
#   • The 4 MB per-file open cost is ~43% of the weighted size because
#     there are many small files; compacting them would reduce it.
#   • More cores -> smaller target split size -> more partitions.
# ------------------------------------------------------------------
print("maxPartitionBytes :", spark.conf.get("spark.sql.files.maxPartitionBytes"))
print("openCostInBytes   :", spark.conf.get("spark.sql.files.openCostInBytes"))

**What this cell does:** Shows the first 5 rows of transactions without truncating column values.

**Core concept:** `show()` is an **action**, so it triggers a job. Because only 5 rows are needed, Spark adds a limit (`CollectLimit`) and usually scans only one or a few partitions instead of the whole table, which is why it returns quickly. Note that all columns are stored as strings, including `amt`; casting matters later.

In [0]:
df_transactions.show(5, False)

* A `batch` refers to a group of rows that are processed together.

**What this cell does:** Defines a DataFrame on `customers.parquet`, the small dimension table with one row per customer (`cust_id`, `name`, `age`, `gender`, `birthday`, `zip`, `city`).

**Core concept:** Joining a large **fact** table (transactions) with a small **dimension** table (customers) is the typical pattern where join strategy matters. A small table like this is a candidate for a broadcast join, covered below.

In [0]:
customers_file = f"{DATA_PATH}/customers.parquet"
df_customers = spark.read.parquet(customers_file)

**What this cell does:** Displays the first 5 customer rows to check the schema and values.

**Core concept:** Like any `show()`, this is an action with a limit, so it runs a short job. Note that `age` is a string and `name` holds both first and last name; both details drive the transformations in the next section.

In [0]:
df_customers.show(5, False)

# Narrow Transformations
- `filter` rows where `city='boston'`
- `add` a new column: adding `first_name` and `last_name`
- `alter` an exisitng column: adding 5 to `age` column
- `select` relevant columns

**What this cell does:** Keeps only Boston customers, splits `name` into `first_name` and `last_name`, adds 5 to `age`, selects the final columns, and runs the plan with the `noop` writer.

**Core concept: narrow transformations.** In a narrow transformation each output partition depends on exactly one input partition, so **no data moves between executors (no shuffle)**. `filter`, `withColumn` and `select` are all narrow. Spark chains them into a **single stage** and fuses them with whole-stage code generation, so in the DAG you see one box: Scan parquet, Filter, Project. The optimizer also pushes `city = 'boston'` into the parquet scan (`PushedFilters`) and reads only the needed columns.

**The `noop` format** runs the full query, reading and transforming every row, but discards the output. It is a standard way to benchmark a plan or study its DAG without the cost of writing files.

In [0]:
df_customers.select(F.spark_partition_id().alias("pid")).distinct().count()

In [0]:
df_customers.count()

In [0]:
df_narrow_transform = (
    df_customers
    .filter(F.col("city") == "boston")
    .withColumn("first_name", F.split("name", " ").getItem(0))
    .withColumn("last_name", F.split("name", " ").getItem(1))
    .withColumn("age", F.col("age") + F.lit(5))
    .select("cust_id", "first_name", "last_name", "age", "gender", "birthday")
)

df_narrow_transform.write.format("noop").mode("overwrite").save()

**What this cell does:** Shows 7 rows of the transformed DataFrame so you can check the new name columns and the increased age.

**Core concept:** DataFrames are not stored after an action. This `show()` re-executes the plan from the source files (only as much as needed for 7 rows), because nothing is cached. Adding 5 to the string `age` implicitly casts it to a number, which is why the output type is `long`.

In [0]:
df_narrow_transform.show(7, False)

**What this cell does:** Filters customers older than 50 and runs the plan with the `noop` writer.

**Core concept:** This is another single-stage, narrow plan. `age` is stored as a string, so it is cast to `int` before the comparison; comparing strings would be lexicographic (`"9" > "50"`). Because the predicate wraps the column in a cast, Spark generally cannot push it down into the parquet reader as a simple filter, so it is evaluated after the scan. Compare this with the pushed-down `city = 'boston'` filter in the Spark UI.

In [0]:
df_customer_gt_50 = (
    df_customers
    .filter(F.col("age").cast("int") > 50)
)
df_customer_gt_50.write.format("noop").mode("overwrite").save()

# Wide Transformations
1. Joins
   - Sort Merge Join
   - Broadcast Join
2. GroupBy
   - `count`
   - `countDistinct`
   - `sum`

**Core concept: wide transformations.** In a wide transformation an output partition needs rows from **many** input partitions, for example all rows with the same `cust_id` or `city`. Spark has to **shuffle**: each task writes its rows, hash-partitioned by key, to local disk, and tasks in the next stage fetch their slice over the network.

In the DAG a shuffle appears as an **Exchange** node and always creates a **new stage**. Shuffles are usually the most expensive part of a query because of disk I/O, network transfer and serialization. They are also where data skew shows up: one hot key produces one huge partition and one slow task. `spark.sql.shuffle.partitions` (default 200) sets the number of post-shuffle partitions, and Adaptive Query Execution (AQE, on by default in Databricks) can coalesce or split them at runtime.

## 1. Joins

### Sort Merge Join

**What this cell does:** Sets `spark.sql.autoBroadcastJoinThreshold` to `-1`, which turns off automatic broadcast joins so Spark falls back to a sort-merge join.

**Core concept: sort-merge join (SMJ).** SMJ is Spark's default strategy for joining two large tables:
1. **Shuffle** both sides by the join key (`cust_id`), so matching keys land in the same partition.
2. **Sort** each partition by the key.
3. **Merge**: walk both sorted streams together and emit matching pairs.

It scales to any size because nothing has to fit in memory as a whole, but it pays for two shuffles and two sorts. Note: on Databricks, AQE can still turn an SMJ into a broadcast join at runtime once it sees the actual table sizes (controlled by `spark.sql.adaptive.autoBroadcastJoinThreshold`). If the DAG shows a broadcast here, that is why.

In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

**What this cell does:** Defines an inner join of transactions with customers on `cust_id`. No data is processed yet; the join is lazy.

**Core concept:** Joining with `on="cust_id"` (a column name rather than an expression) keeps a single `cust_id` column in the result. Both tables also have a `city` column, so the result contains **two** `city` columns. Selecting `city` later would be ambiguous, so in real code rename or drop one of them.

In [0]:
df_joined = (
    df_transactions.join(
        df_customers,
        how="inner",
        on="cust_id"
    )
)

**What this cell does:** Runs the join plan end to end with the `noop` writer.

**Core concept:** Open the SQL / DataFrame DAG for this run. With a sort-merge join you should see two scan branches, each followed by **Exchange hashpartitioning(cust_id, ...)** and **Sort**, meeting in a **SortMergeJoin** node. That means at least three stages: one per side to write shuffle files, plus the join stage. Check the shuffle read and write sizes and task durations; because this dataset comes from a `data_skew` folder, a few `cust_id` values may have far more rows, which makes some join tasks much slower than others. AQE skew-join handling may split those partitions.

In [0]:
df_joined.write.format("noop").mode("overwrite").save()

### Broadcast Join

**What this cell does:** Restores the auto-broadcast threshold to 10 MB (10485760 bytes), the Spark default.

**Core concept: broadcast hash join (BHJ).** If one side of a join is small enough, Spark collects it to the driver, builds a hash table, and **broadcasts** a copy to every executor. Each task of the large table then probes the local hash table, so the **large table is never shuffled**. Spark picks a BHJ automatically when its estimated size of a side is below `autoBroadcastJoinThreshold`.

In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)

**What this cell does:** Defines the same inner join, but wraps `df_customers` in `F.broadcast(...)`.

**Core concept:** `F.broadcast` is a **join hint** that tells Spark to broadcast this side whatever its size estimate or threshold. Use it when you know a table is small. The trade-offs: the broadcast table must fit in driver and executor memory, broadcasts have a hard size limit (8 GB), and a broadcast that is too large can cause out-of-memory errors or long broadcast times.

In [0]:
df_broadcast_joined = (
    df_transactions.join(
        F.broadcast(df_customers),
        how="inner",
        on="cust_id"
    )
)

**What this cell does:** Runs the broadcast join plan with the `noop` writer.

**Core concept:** Compare this DAG with the sort-merge join one. The customers side now goes through a **BroadcastExchange**, and the transactions side flows straight from Scan into **BroadcastHashJoin** with **no Exchange and no Sort**. Fewer stages, no shuffle of the 40M-row table and no skewed join partitions usually make this much faster. Compare the durations of the two runs.

In [0]:
df_broadcast_joined.write.format("noop").mode("overwrite").save()

## 2. GroupBy

### GroupBy Count

**What this cell does:** Defines a count of transactions per `city` (lazy; it runs at the next `show`).

**Core concept: two-phase aggregation.** `groupBy` is a wide transformation, but Spark makes it cheap with **partial aggregation**:
1. **HashAggregate (partial)**: each task counts rows per city within its own partition (a map-side combine).
2. **Exchange hashpartitioning(city)**: only these small partial counts are shuffled, not the raw rows.
3. **HashAggregate (final)**: partial counts for each city are added together.

Because there are only a few cities, the shuffle is tiny even though the input has millions of rows.

In [0]:
df_city_counts = (
    df_transactions
    .groupBy("city")
    .count()
)

**What this cell does:** Triggers the aggregation and shows 5 cities with their transaction counts.

**Core concept:** In the DAG, look for the partial HashAggregate, Exchange and final HashAggregate pattern, and notice how few rows cross the Exchange. The counts are about 3.98M per city, so the data is evenly spread across cities (the skew in this dataset is on `cust_id`, not `city`). Group order is not guaranteed without an `orderBy`.

In [0]:
df_city_counts.show(5, False)

**What this cell does:** Defines the total transaction amount (`sum(amt)`) per city.

**Core concept:** This uses the same two-phase plan as `count`, but uses `agg()` so the result column can be named. `amt` is stored as a string, so Spark implicitly casts it to `double` before summing. In real code, cast explicitly, ideally to `decimal` for money.

In [0]:
df_txn_amt_city = (
    df_transactions
    .groupBy("city")
    .agg(F.sum("amt").alias("txn_amt"))
)

**What this cell does:** Triggers the sum aggregation and shows 5 cities with their totals.

**Core concept:** The last digits of these totals can change between runs (and differ from the Jupyter run). Floating-point addition is not associative, and the order in which partial sums are combined depends on partitioning and task timing. Using `decimal` gives exact, repeatable totals.

In [0]:
df_txn_amt_city.show(5, False)

### GroupBy Count Distinct 

**What this cell does:** Defines the number of distinct `txn_id` values per city.

**Core concept:** A distinct count cannot be combined from partial counts: the same `txn_id` might appear in several partitions. Spark therefore plans extra aggregation steps. It first de-duplicates on `(city, txn_id)` (partial aggregate, then an **Exchange on (city, txn_id)**, then a merge), and then counts per city with another partial and final aggregate. Because `txn_id` has very high cardinality, far more data is shuffled than for a plain count, which makes `countDistinct` one of the more expensive aggregations. When an estimate is enough, `F.approx_count_distinct` (HyperLogLog) avoids most of that cost.

In [0]:
df_txn_per_city = (
    df_transactions
    .groupBy("city")
    .agg(F.countDistinct("txn_id").alias("txn_count"))
)

**What this cell does:** Triggers the distinct-count aggregation and shows the results.

**Core concept:** Compare this DAG with the plain `count`: there are more HashAggregate nodes and more Exchanges, and the shuffle is much larger. The distinct counts match the row counts exactly, which shows that every `txn_id` is unique. When a key is known to be unique, a plain `count()` gives the same answer much more cheaply.

In [0]:
df_txn_per_city.show(5, False)

**What this cell does:** Nothing to run. It replaces the original `spark.stop()`.

**Core concept:** In local Jupyter you own the Spark session and stop it to free resources. On Databricks the session belongs to the cluster and is shared with the notebook's lifecycle; stopping it is unnecessary and can break later cells. Resources are freed when the notebook detaches or the cluster auto-terminates.

In [0]:
# spark.stop() is not needed on Databricks; the session is managed by the cluster